# 1. Préambule

## 1.1 Problématique

La très jeune start-up de l'AgriTech, nommée "**Fruits**!", <br />
cherche à proposer des solutions innovantes pour la récolte des fruits.

La volonté de l’entreprise est de préserver la biodiversité des fruits <br />
en permettant des traitements spécifiques pour chaque espèce de fruits <br />
en développant des robots cueilleurs intelligents.

La start-up souhaite dans un premier temps se faire connaître en mettant <br />
à disposition du grand public une application mobile qui permettrait aux <br />
utilisateurs de prendre en photo un fruit et d'obtenir des informations sur ce fruit.

Pour la start-up, cette application permettrait de sensibiliser le grand public <br /> 
à la biodiversité des fruits et de mettre en place une première version du moteur <br />
de classification des images de fruits.

De plus, le développement de l’application mobile permettra de construire <br />
une première version de l'architecture **Big Data** nécessaire.

## 1.2 Objectifs dans ce projet

1. Développer une première chaîne de traitement des données qui <br />
   comprendra le **preprocessing** et une étape de **réduction de dimension**.
2. Tenir compte du fait que <u>le volume de données va augmenter <br />
   très rapidement</u> après la livraison de ce projet, ce qui implique de:
 - Déployer le traitement des données dans un environnement **Big Data**
 - Développer les scripts en **pyspark** pour effectuer du **calcul distribué**

## 1.3 Déroulement des étapes du projet

Le projet va être réalisé en 2 temps, dans deux environnements différents. <br />
Nous allons dans un premier temps développer et exécuter notre code en local, <br />
en travaillant sur un nombre limité d'images à traiter.

Une fois les choix techniques validés, nous déploierons notre solution <br />
dans un environnement Big Data en mode distribué.

## 1.4 Transfert Learning

L'énoncé du projet nous demande également de <br />
réaliser une première chaîne de traitement <br />
des données qui comprendra le preprocessing et <br />
une étape de réduction de dimension.

Il est également précisé qu'il n'est pas nécessaire <br />
d'entraîner un modèle pour le moment.

Nous décidons de partir sur une solution de **transfert learning**.

Simplement, le **transfert learning** consiste <br />
à utiliser la connaissance déjà acquise <br />
par un modèle entraîné (ici **MobileNetV2**) pour <br />
l'adapter à notre problématique.

Nous allons fournir au modèle nos images, et nous allons <br />
<u>récupérer l'avant dernière couche</u> du modèle.<br />
En effet la dernière couche de modèle est une couche softmax <br />
qui permet la classification des images ce que nous ne <br />
souhaitons pas dans ce projet.

L'avant dernière couche correspond à un **vecteur <br />
réduit** de dimension (1,1,1280).

Cela permettra de réaliser une première version du moteur <br />
pour la classification des images des fruits.

**MobileNetV2** a été retenu pour sa <u>rapidité d'exécution</u>, <br />
particulièrement adaptée pour le traitement d'un gros volume <br />
de données ainsi que la <u>faible dimensionnalité du vecteur <br />
de caractéristique en sortie</u> (1,1,1280)

# 2. Déploiement de la solution dans le cloud

## 2.1 Démarrage de la session Spark

In [1]:
# L'exécution de cette cellule démarre l'application Spark

VBox()

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
0,application_1782476915954_0001,pyspark,idle,Link,Link,None,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

<u>Affichage des informations de Spark en cours d'execution</u> :

In [43]:
spark

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

<u>Affichage des informations sur la session en cours et liens vers Spark UI</u> :

In [3]:
%%info

ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
2,application_1782742894642_0003,pyspark,idle,Link,Link,None,✔


## 2.2 Installation des packages

Les packages nécessaires ont été installé via l'étape de **bootstrap** à l'instanciation du serveur.

In [6]:
sc.install_pypi_package("tensorflow-cpu==2.15.1")

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

  Attempting uninstall: cffi
    Found existing installation: cffi 1.14.5
    Not uninstalling cffi at /usr/lib64/python3.9/site-packages, outside environment /mnt/yarn/usercache/livy/appcache/application_1782742894642_0003/container_1782742894642_0003_01_000002/tmp/spark-651e4af1-a4c4-4de3-b80f-42bf02f368a3
    Can't uninstall 'cffi'. No files were found to uninstall.
  Attempting uninstall: cryptography
    Found existing installation: cryptography 36.0.1
    Not uninstalling cryptography at /usr/lib64/python3.9/site-packages, outside environment /mnt/yarn/usercache/livy/appcache/application_1782742894642_0003/container_1782742894642_0003_01_000002/tmp/spark-651e4af1-a4c4-4de3-b80f-42bf02f368a3
    Can't uninstall 'cryptography'. No files were found to uninstall.
  Attempting uninstall: markupsafe
    Found existing installation: MarkupSafe 1.1.1
    Not uninstalling markupsafe at /usr/lib64/python3.9/site-packages, outside environment /mnt/yarn/usercache/livy/appcache/application_17

## 2.3 Import des librairies

In [11]:
import pandas as pd
import numpy as np
import io
import os
import tensorflow as tf
from PIL import Image
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2, preprocess_input
from tensorflow.keras.preprocessing.image import img_to_array
from tensorflow.keras import Model
from pyspark.sql.functions import col, pandas_udf, PandasUDFType, element_at, split, udf
from pyspark.sql import SparkSession
from pyspark.ml.feature import PCA
from pyspark.ml.linalg import Vectors, VectorUDT

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

## 2.4 Définition des PATH pour charger les images et enregistrer les résultats

Nous accédons directement à nos **données sur S3** comme si elles étaient **stockées localement**.

In [12]:
PATH = "s3://p9bigdata"
PATH_Data = PATH+'/Test'
PATH_Result = PATH+'/Results'
print('PATH:        '+\
      PATH+'\nPATH_Data:   '+\
      PATH_Data+'\nPATH_Result: '+PATH_Result)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

PATH:        s3://p9bigdata
PATH_Data:   s3://p9bigdata/Test
PATH_Result: s3://p9bigdata/Results

## 2.5 Traitement des données

### 2.5.1 Chargement des données

In [13]:
images = spark.read.format("binaryFile") \
  .option("pathGlobFilter", "*.jpg") \
  .option("recursiveFileLookup", "true") \
  .load(PATH_Data)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [14]:
images.show(5)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+--------------------+-------------------+------+--------------------+
|                path|   modificationTime|length|             content|
+--------------------+-------------------+------+--------------------+
|s3://p9bigdata/Te...|2026-06-22 13:37:42|  7353|[FF D8 FF E0 00 1...|
|s3://p9bigdata/Te...|2026-06-22 13:37:42|  7350|[FF D8 FF E0 00 1...|
|s3://p9bigdata/Te...|2026-06-22 13:37:42|  7349|[FF D8 FF E0 00 1...|
|s3://p9bigdata/Te...|2026-06-22 13:37:42|  7348|[FF D8 FF E0 00 1...|
|s3://p9bigdata/Te...|2026-06-22 13:37:43|  7328|[FF D8 FF E0 00 1...|
+--------------------+-------------------+------+--------------------+
only showing top 5 rows

<u>Je ne conserve que le **path** de l'image et j'ajoute <br />
    une colonne contenant les **labels** de chaque image</u> :

In [15]:
images = images.withColumn('label', element_at(split(images['path'], '/'),-2))
print(images.printSchema())
print(images.select('path','label').show(5,False))

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

root
 |-- path: string (nullable = true)
 |-- modificationTime: timestamp (nullable = true)
 |-- length: long (nullable = true)
 |-- content: binary (nullable = true)
 |-- label: string (nullable = true)

None
+--------------------------------------------+----------+
|path                                        |label     |
+--------------------------------------------+----------+
|s3://p9bigdata/Test/Watermelon/r_106_100.jpg|Watermelon|
|s3://p9bigdata/Test/Watermelon/r_109_100.jpg|Watermelon|
|s3://p9bigdata/Test/Watermelon/r_108_100.jpg|Watermelon|
|s3://p9bigdata/Test/Watermelon/r_107_100.jpg|Watermelon|
|s3://p9bigdata/Test/Watermelon/r_95_100.jpg |Watermelon|
+--------------------------------------------+----------+
only showing top 5 rows
None

### 2.5.2 Préparation du modèle

In [16]:
model = MobileNetV2(weights='imagenet',
                    include_top=True,
                    input_shape=(224, 224, 3))

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

14536120/14536120 [==============================] - 1s 0us/step

In [17]:
new_model = Model(inputs=model.input,
                  outputs=model.layers[-2].output)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [18]:
broadcast_weights = sc.broadcast(new_model.get_weights())

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [19]:
new_model.summary()

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_1 (InputLayer)        [(None, 224, 224, 3)]        0         []                            
                                                                                                  
 Conv1 (Conv2D)              (None, 112, 112, 32)         864       ['input_1[0][0]']             
                                                                                                  
 bn_Conv1 (BatchNormalizati  (None, 112, 112, 32)         128       ['Conv1[0][0]']               
 on)                                                                                              
                                                                                                  
 Conv1_relu (ReLU)           (None, 112, 112, 32)         0         ['bn_Conv1[0][0]']        

In [20]:
def model_fn():
    """
    Returns a MobileNetV2 model with top layer removed 
    and broadcasted pretrained weights.
    """
    model = MobileNetV2(weights='imagenet',
                        include_top=True,
                        input_shape=(224, 224, 3))
    for layer in model.layers:
        layer.trainable = False
    new_model = Model(inputs=model.input,
                  outputs=model.layers[-2].output)
    new_model.set_weights(broadcast_weights.value)
    return new_model

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

### 2.5.3 Définition du processus de chargement des images <br/> et application de leur featurisation à travers l'utilisation de pandas UDF

In [21]:
def preprocess(content):
    """
    Preprocesses raw image bytes for prediction.
    """
    img = Image.open(io.BytesIO(content)).resize([224, 224])
    arr = img_to_array(img)
    return preprocess_input(arr)

def featurize_series(model, content_series):
    """
    Featurize a pd.Series of raw images using the input model.
    :return: a pd.Series of image features
    """
    input = np.stack(content_series.map(preprocess))
    preds = model.predict(input)
    # For some layers, output features will be multi-dimensional tensors.
    # We flatten the feature tensors to vectors for easier storage in Spark DataFrames.
    output = [p.flatten() for p in preds]
    return pd.Series(output)

@pandas_udf('array<float>', PandasUDFType.SCALAR_ITER)
def featurize_udf(content_series_iter):
    '''
    This method is a Scalar Iterator pandas UDF wrapping our featurization function.
    The decorator specifies that this returns a Spark DataFrame column of type ArrayType(FloatType).

    :param content_series_iter: This argument is an iterator over batches of data, where each batch
                              is a pandas Series of image data.
    '''
    # With Scalar Iterator pandas UDFs, we can load the model once and then re-use it
    # for multiple data batches.  This amortizes the overhead of loading big models.
    model = model_fn()
    for content_series in content_series_iter:
        yield featurize_series(model, content_series)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

/mnt/yarn/usercache/livy/appcache/application_1782742894642_0003/container_1782742894642_0003_01_000002/pyspark.zip/pyspark/sql/pandas/functions.py:446: UserWarning: In Python 3.6+ and Spark 3.0+, it is preferred to specify type hints for pandas UDF instead of specifying pandas UDF type which will be deprecated in the future releases. See SPARK-28264 for more details.

### 2.6.4 Exécutions des actions d'extractions de features

In [22]:
# spark.conf.set("spark.sql.execution.arrow.maxRecordsPerBatch", "1024")

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [23]:
features_df = images.repartition(24).select(col("path"),
                                            col("label"),
                                            featurize_udf("content").alias("features")
                                           )

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [24]:
print(PATH_Result)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

s3://p9bigdata/Results

### 2.5.5 Réduction de dimension des features par PCA

La bibliothèque Spark ML exige un objet VectorUDT pour les algorithmes de Machine Learning.  
  
Une étape de conversion array<float> → VectorUDT est donc nécessaire <br />
avant d'appliquer la PCA.

<u>Créer une fonction Spark qui prend un tableau (arr) et le transforme en vecteur dense Spark ML</u>.

In [25]:
array_to_vector_udf = udf(lambda arr: Vectors.dense(arr), VectorUDT())

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

<u>Créer un nouveau DataFrame contenant une colonne supplémentaire appelée features_vector

In [26]:
features_vectorized_df = features_df.withColumn(
    "features_vector",
    array_to_vector_udf(col("features"))
)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [27]:
features_vectorized_df.dtypes

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

[('path', 'string'), ('label', 'string'), ('features', 'array<float>'), ('features_vector', 'vector')]

Vérification des types. La colonne "features_vector" est bien un vecteurUDF  
C'est cette colonne que nous utiliserons pour la PCA. 

In [28]:
features_vectorized_df.dtypes

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

[('path', 'string'), ('label', 'string'), ('features', 'array<float>'), ('features_vector', 'vector')]

<u>Détermination du nombre de composantes PCA </u>:
   
Avec Spark ML, il faut obligatoirement fournir le nombre de composantes principales `k` à conserver.   
Afin de choisir une valeur pertinente, une première PCA est entraînée avec un nombre élevé de composantes.  
La variance expliquée par chaque composante est ensuite analysée à l'aide de l'attribut :
```python
model_pca.explainedVariance
```

La variance expliquée cumulée est calculée dans le but de mesurer la quantité d'information conservée <br />
lorsque le nombre de composantes augmente.

```python
cum_var = np.cumsum(model_pca.explainedVariance)
```

Une lecture de variance expliquée cumulée permet alors d'identifier le nombre minimal de composantes<br />
conservant une part suffisante de l'information.

Le nombre de composantes retenu correspond ainsi à un compromis entre :  

- la réduction de la dimension des données ;
- la conservation de l'information contenue dans les vecteurs de caractéristiques ;
- les performances de stockage et de calcul.

In [29]:
features_vectorized_df = features_vectorized_df.cache()
features_vectorized_df.count()

pca_var = PCA(k=200,
          inputCol="features_vector",
          outputCol="pca_features"
         )
model_pca_var = pca_var.fit(features_vectorized_df)

explained_var = model_pca_var.explainedVariance
cum_var = np.cumsum(explained_var)
for i in np.arange(50, 200, 20):
    print(i, cum_var[i-1])

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

50 0.7318304612940538
70 0.7828701302858415
90 0.8174394608169627
110 0.8430177368240853
130 0.8628182875829454
150 0.8786387705145982
170 0.8916764153515365
190 0.9025791535276227

In [ ]:
seuil = 0.90
k_optimal = np.argmax(cum_var >= seuil) + 1
print(
    f"{seuil:.0%} de variance expliquée atteinte avec "
    f"{k_optimal} composantes"
)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

90% de variance expliquee atteints avec 185 composantes

Nous choisissons k=k_optimal qui contient seuil% de la variance.  
Cela représente une réduction significative de plus de 85% de la dimensions <br />
tout en conservant la 90% de l'information.

In [35]:
pca = PCA(k=k_optimal,
          inputCol="features_vector",
          outputCol="pca_features"
         ) # Crée une PCA qui réduit les 1280 dimensions à k_optimal Composantes Principales

# Entraîne la PCA sur les vecteurs de la colonne features.
model_pca = pca.fit(features_vectorized_df)

# Appliquation la PCA au DataFrame
features_pca_df = model_pca.transform(features_vectorized_df)

# affiche le vecteur PCA de la première ligne.
# features_pca_df.select("pca_features").show(1, truncate=False)

# Donne la part de variance expliquée par chaque composante principale.
# model_pca.explainedVariance 


VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

<u>En option, la sauvegarde de l'objet et du modèle pca </u>:  
(cellule en RAW)

## 2.6 Enregistrement des données traitées au format "**parquet**"</u> :

In [36]:
features_pca_df.select("path", "label", "features", "pca_features") \
    .write.mode("overwrite") \
    .parquet(PATH_Result)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

## 2.7 Chargement des données enregistrées et validation du résultat

In [37]:
df = pd.read_parquet(PATH_Result, engine='pyarrow')

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

<u>On affiche les 5 premières lignes du DataFrame</u> :

In [38]:
df.head()

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

                                            path  ...                                       pca_features
0    s3://p9bigdata/Test/Watermelon/r_89_100.jpg  ...  {'type': 1, 'size': None, 'indices': None, 'va...
1    s3://p9bigdata/Test/Watermelon/r_94_100.jpg  ...  {'type': 1, 'size': None, 'indices': None, 'va...
2    s3://p9bigdata/Test/Watermelon/r_96_100.jpg  ...  {'type': 1, 'size': None, 'indices': None, 'va...
3  s3://p9bigdata/Test/Pineapple Mini/33_100.jpg  ...  {'type': 1, 'size': None, 'indices': None, 'va...
4    s3://p9bigdata/Test/Watermelon/r_54_100.jpg  ...  {'type': 1, 'size': None, 'indices': None, 'va...

[5 rows x 4 columns]

La colonne "pca_features" contient des dictionnaires  
Nous allons récupérer uniquement la liste de vecteurs classés dans "values"  

In [39]:
df["pca_features"]=df["pca_features"].apply(lambda x: x["values"])
df.head()

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

                                            path  ...                                       pca_features
0    s3://p9bigdata/Test/Watermelon/r_89_100.jpg  ...  [-2.604317122623734, 5.529112432642275, -5.521...
1    s3://p9bigdata/Test/Watermelon/r_94_100.jpg  ...  [-2.1068593763339165, 5.6817339768686095, -7.0...
2    s3://p9bigdata/Test/Watermelon/r_96_100.jpg  ...  [-2.4219552741774573, 4.990156620057259, -5.60...
3  s3://p9bigdata/Test/Pineapple Mini/33_100.jpg  ...  [-5.125120576780171, 3.3580061949327504, 1.905...
4    s3://p9bigdata/Test/Watermelon/r_54_100.jpg  ...  [-2.0346489762021136, 4.880721724150277, -5.56...

[5 rows x 4 columns]

<u>On valide la dimension des vecteurs de caractéristiques des images </u> :

In [40]:
print("dimensions après pca : ",df.loc[0,"pca_features"].shape)
print("dimensions avant pca : ",df.loc[0,'features'].shape)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

dimensions apr?s pca :  (185,)
dimensions avant pca :  (1280,)

## 2.8 Export de la réduction de dimension au format csv

In [41]:
df[["path", "label", "pca_features"]].to_csv(PATH_Result+'/pca_features.csv', index=False)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

# 3. Conclusion

Nous avons réalisé ce projet **en deux temps** en tenant <br />
compte des contraintes qui nous ont été imposées.

Nous avons **dans un premier temps développé notre solution en local** <br />
sur une machine virtuelle dans un environnement Linux Ubuntu.

La <u>première phase</u> a consisté à **installer l'environnement de travail Spark**. <br />
**Spark** a un paramètre qui nous permet de travaillé en local et nous permet <br />
ainsi de **simuler du calcul partagé** en considérant <br />
**chaque cœur d'un processeur comme un worker indépendant**.<br />
Nous avons travaillé sur un plus **petit jeu de donnée**, l'idée était <br />
simplement de **valider le bon fonctionnement de la solution**.

Nous avons fait le choix de réaliser du **transfert learning** <br />
à partir du model **MobileNetV2**.<br />
Ce modèle a été retenu pour sa **légèreté** et sa **rapidité d'exécution** <br />
ainsi que pour la **faible dimension de son vecteur en sortie**.

Les résultats ont été enregistrés sur disque en plusieurs <br />
partitions au format "**parquet**".

<u>**La solution a parfaitement fonctionné en mode local**</u>.

La <u>deuxième phase</u> a consisté à créer un **réel cluster de calculs**. <br />
L'objectif était de pouvoir **anticiper une future augmentation de la charge de travail**.

Le meilleur choix retenu a été l'utilisation du prestataire de services **Amazon Web Services** <br />
qui nous permet de **louer à la demande de la puissance de calculs**, <br />
pour un **coût tout à fait acceptable**.<br />
Ce service se nomme **EC2** et se classe parmi les offres **Infrastructure As A Service** (IAAS).

Nous sommes allez plus loin en utilisant un service de plus <br />
haut niveau (**Plateforme As A Service** PAAS)<br />
en utilisant le service **EMR** qui nous permet d'un seul coup <br />
d'**instancier plusieurs serveur (un cluster)** sur lesquels <br />
nous avons pu demander l'installation et la configuration de plusieurs<br />
programmes et librairies nécessaires à notre projet comme **Spark**, <br />
 **JupyterHub** ainsi que la librairie **TensorFlow**.

En plus d'être plus **rapide et efficace à mettre en place**, nous avons <br />
la **certitude du bon fonctionnement de la solution**, celle-ci ayant été <br />
préalablement validé par les ingénieurs d'Amazon.

Nous avons également pu installer, sans difficulté, **les packages <br />
nécessaires sur l'ensembles des machines du cluster**.

Enfin, avec très peu de modification, et plus simplement encore, <br />
nous avons pu **exécuter notre notebook comme nous l'avions fait localement**.<br />
Nous avons cette fois-ci exécuté le traitement sur **l'ensemble des images de notre dossier "Test"**.

Nous avons opté pour le service **Amazon S3** pour **stocker les données de notre projet**. <br />
S3 offre, pour un faible coût, toutes les conditions dont nous avons besoin pour stocker <br />
et exploiter de manière efficace nos données.<br />
L'espace alloué est potentiellement **illimité**, mais les coûts seront fonction de l'espace utilisé.

Il nous sera **facile de faire face à une monté de la charge de travail** en **redimensionnant** <br />
simplement notre cluster de machines (horizontalement et/ou verticalement au besoin), <br />
les coûts augmenteront en conséquence mais resteront nettement inférieurs aux coûts engendrés <br />
par l'achat de matériels ou par la location de serveurs dédiés.